# Week 2: Job Bank Data Understanding and Data Quality Assessment

## Objective

The objective of this notebook is to understand the structure and quality of the Job Bank dataset, identify potential data quality issues, and further prepare the dataset for descriptive analysis as well as occupation selection in subsequent project stages.

This notebook includes:
- Dataset overview
- Data type assessment
- Missing value analysis
- Duplicate record analysis
- Date coverage assessmen
- Geographic coverage assessment
- NOC coverage assessment
- Wage unit analysis
- Vacancy count analysis
- Education and experience requirement analysis
- Employment type analysis
- Data cleaning and preparation

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [7]:
from pathlib import Path

# Define project root relative to notebook location

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "Data" / "Raw_Data"

OUTPUTS = PROJECT_ROOT / "Outputs"
TABLES = OUTPUTS / "Tables"
FIGURES = OUTPUTS / "Figures"
REPORTS = OUTPUTS / "Reports"

# Create output folders if they do not already exist

for folder in [TABLES, FIGURES, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

In [9]:
# Load Job Bank raw dataset

JOB_BANK_FILE = (
    RAW_DATA /
    "job_bank-open-data-all-job-postings-june-2026.csv"
)

job_bank = pd.read_csv(
    JOB_BANK_FILE,
    low_memory=False
)

print(
    f"Dataset contains "
    f"{job_bank.shape[0]:,} rows and "
    f"{job_bank.shape[1]} columns."
)

Dataset contains 55,091 rows and 65 columns.


In [11]:
# Create a working copy and standardize column names

job_bank_clean = job_bank.copy()

job_bank_clean.columns = (
    job_bank_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^\w]+", "_", regex=True)
    .str.strip("_")
)

print(job_bank_clean.columns.tolist())

['ythwic_job_location_snapshot_id', 'job_title', 'original_job_title', 'noc_2016_code', 'noc_2016_code_name', 'noc21_code', 'noc21_code_name', 'external_indicator', 'first_posting_date', 'vacancy_count', 'official_language', 'education_los', 'experience_level', 'government_type', 'placement_agency', 'naics', 'province_territory', 'city', 'work_location_postal_code', 'economic__region', 'various_location', 'employment_type', 'employment_term', 'employment_term_start_date', 'employment_term_end_date', 'employment_term_oncall', 'employment_term_overtime', 'employment_term_day', 'employment_term_evening', 'employment_term_shift', 'employment_term_weekend', 'employment_term_night', 'employment_term_telework', 'employment_term_early', 'employment_term_flex', 'employment_term_morning', 'employment_term_tbd', 'salary_condition_detail', 'salary_per', 'salary_minimum', 'salary_maximum', 'salary_condition_collective', 'salary_condition_bonus', 'salary_condition_disability', 'salary_condition_grat

In [13]:
# Map raw Job Bank column names to standardized analytical names

column_mapping = {
    "ythwic_job_location_snapshot_id": "posting_id",
    "job_title": "job_title",
    "original_job_title": "original_job_title",
    "noc_2016_code": "noc_2016_code",
    "noc_2016_code_name": "noc_2016_name",
    "noc21_code": "noc_2021_code",
    "noc21_code_name": "noc_2021_name",
    "first_posting_date": "posting_date",
    "vacancy_count": "vacancy_count",
    "education_los": "education_level",
    "experience_level": "experience_level",
    "province_territory": "province",
    "city": "city",
    "economic_region": "economic_region",
    "employment_type": "employment_type",
    "employment_term": "employment_term",
    "salary_per": "salary_period",
    "salary_minimum": "salary_minimum",
    "salary_maximum": "salary_maximum",
    "hours_per": "hours_per",
    "hours_minimum": "hours_minimum",
    "hours_maximum": "hours_maximum",
    "work_hours": "work_hours"
}

In [15]:
# Rename columns using standardized analytical names

job_bank_clean = job_bank_clean.rename(
    columns=column_mapping
)

In [17]:
# Preview the cleaned dataset after column standardization

job_bank_clean.head()

,posting_id,job_title,original_job_title,noc_2016_code,noc_2016_name,noc_2021_code,noc_2021_name,external_indicator,posting_date,vacancy_count,...,condition_vision_care,salary_condition_other_benefits,commission_per,commission_type,hours_per,hours_minimum,hours_maximum,work_hours,work_hours_from_time,work_hours_to_time
0,15645194,Facilities and Fleet Maintenance Support,NaN,NaN,NaN,NaN,NaN,1,2026/06/17,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
1,15675531,sales associate,Part Time Sales Associate-SoftMoc ShoeRack Cro...,6421.0,Retail salespersons,64100.0,Retail salespersons and visual merchandisers,1,2026/06/29,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
2,15597307,"testing, adjusting and balancing (tab) technic...",HVAC R & Gasfitter Journeymen Apprentice (Leve...,2232.0,Mechanical engineering technologists and techn...,11201.0,Professional occupations in business managemen...,1,2026/06/01,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN
3,15621351,"chartered professional accountant, chartered a...",NaN,1111.0,Financial auditors and accountants,11100.0,Financial auditors and accountants,0,2026/06/09,1,...,No,No,NaN,NaN,Week,4.0,10.0,No,NaN,NaN
4,15629538,refrigeration technician,NaN,7313.0,Refrigeration and air conditioning mechanics,72402.0,"Heating, refrigeration and air conditioning me...",1,2026/06/11,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No,NaN,NaN


In [19]:
# Assess data types of all variables

data_types = pd.DataFrame({
    "Column": job_bank_clean.columns,
    "Data_Type": job_bank_clean.dtypes.astype(str).values
})

data_types

,Column,Data_Type
0,posting_id,int64
1,job_title,object
2,original_job_title,object
3,noc_2016_code,float64
4,noc_2016_name,object
...,...,...
60,hours_minimum,float64
61,hours_maximum,float64
62,work_hours,object
63,work_hours_from_time,object


In [21]:
# Save data type assessment table

data_types.to_csv(
    TABLES / "job_bank_data_types.csv",
    index=False
)

In [23]:
# Analyze missing values across dataset columns

missing_table = pd.DataFrame({
    "Column": job_bank_clean.columns,
    "Missing_Count": job_bank_clean.isna().sum().values,
    "Missing_Percentage": (
        job_bank_clean.isna().mean() * 100
    ).round(2).values
})

missing_table = missing_table.sort_values(
    "Missing_Percentage",
    ascending=False
)

missing_table.head(20)

,Column,Missing_Count,Missing_Percentage
58,commission_type,54894,99.64
57,commission_per,54894,99.64
24,employment_term_end_date,54660,99.22
64,work_hours_to_time,54209,98.40
63,work_hours_from_time,54209,98.40
23,employment_term_start_date,49111,89.15
2,original_job_title,33737,61.24
13,government_type,33369,60.57
44,salary_condition_gratuity,33112,60.10
43,salary_condition_disability,33112,60.10


In [25]:
# Save missing value analysis table

missing_table.to_csv(
    TABLES / "job_bank_missing_value_table.csv",
    index=False
)

In [27]:
# Assess duplicate records in the dataset

duplicate_count = job_bank_clean.duplicated().sum()

duplicate_percentage = (
    duplicate_count /
    len(job_bank_clean)
) * 100

print("Duplicate rows:", duplicate_count)
print(
    f"Duplicate percentage: "
    f"{duplicate_percentage:.2f}%"
)

Duplicate rows: 0
Duplicate percentage: 0.00%


In [29]:
# Create duplicate record summary table

duplicate_analysis = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Duplicate Rows",
        "Duplicate Percentage"
    ],
    "Value": [
        len(job_bank_clean),
        duplicate_count,
        round(duplicate_percentage, 2)
    ]
})

duplicate_analysis

,Metric,Value
0,Total Rows,55091.0
1,Duplicate Rows,0.0
2,Duplicate Percentage,0.0


In [31]:
# Save duplicate record analysis table

duplicate_analysis.to_csv(
    TABLES / "job_bank_duplicate_analysis.csv",
    index=False
)

In [33]:
# Convert posting date column to datetime format

job_bank_clean["posting_date"] = pd.to_datetime(
    job_bank_clean["posting_date"],
    errors="coerce"
)

In [35]:
# Create date coverage summary table

date_coverage = pd.DataFrame({
    "Metric": [
        "Earliest Posting Date",
        "Latest Posting Date",
        "Missing Posting Dates"
    ],
    "Value": [
        job_bank_clean["posting_date"].min(),
        job_bank_clean["posting_date"].max(),
        job_bank_clean["posting_date"].isna().sum()
    ]
})

date_coverage

,Metric,Value
0,Earliest Posting Date,2026-06-01 00:00:00
1,Latest Posting Date,2026-06-30 00:00:00
2,Missing Posting Dates,0


In [37]:
# Analyze job posting distribution by province

province_summary = (
    job_bank_clean["province"]
    .value_counts(dropna=False)
    .reset_index()
)

province_summary.columns = [
    "Province",
    "Posting_Count"
]

province_summary

,Province,Posting_Count
0,Ontario,16491
1,Québec,10593
2,British Columbia,9937
3,Alberta,6409
4,Saskatchewan,3874
5,Nova Scotia,2880
6,Manitoba,1595
7,New Brunswick,1565
8,Newfoundland and Labrador,1090
9,Prince Edward Island,313


In [39]:
# Count unique provinces and territories represented in the dataset

number_of_provinces = (
    job_bank_clean["province"]
    .nunique()
)

print(
    "Number of provinces/territories:",
    number_of_provinces
)

Number of provinces/territories: 13


In [41]:
# Assess occupation classification coverage using NOC systems

noc_coverage = pd.DataFrame({
    "NOC_System": [
        "NOC 2016",
        "NOC 2021"
    ],
    "Unique_Codes": [
        job_bank_clean["noc_2016_code"].nunique(),
        job_bank_clean["noc_2021_code"].nunique()
    ]
})

noc_coverage

,NOC_System,Unique_Codes
0,NOC 2016,483
1,NOC 2021,500


In [43]:
# Examine vacancy count distribution

job_bank_clean["vacancy_count"].describe()

count    55091.000000
mean         1.616562
std          5.281202
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        999.000000
Name: vacancy_count, dtype: float64

In [45]:
# Check missing values in vacancy count

print(
    "Missing vacancy values:",
    job_bank_clean["vacancy_count"].isna().sum()
)

Missing vacancy values: 0


In [47]:
# Convert vacancy count to numeric format

job_bank_clean["vacancy_count"] = pd.to_numeric(
    job_bank_clean["vacancy_count"],
    errors="coerce"
)

In [49]:
# Create vacancy count summary table

vacancy_summary = pd.DataFrame({
    "Metric": [
        "Total Advertised Vacancies",
        "Mean Vacancies per Posting",
        "Median Vacancies per Posting",
        "Minimum Vacancies",
        "Maximum Vacancies"
    ],
    "Value": [
        job_bank_clean["vacancy_count"].sum(),
        job_bank_clean["vacancy_count"].mean(),
        job_bank_clean["vacancy_count"].median(),
        job_bank_clean["vacancy_count"].min(),
        job_bank_clean["vacancy_count"].max()
    ]
})

vacancy_summary

,Metric,Value
0,Total Advertised Vacancies,89058.000000
1,Mean Vacancies per Posting,1.616562
2,Median Vacancies per Posting,1.000000
3,Minimum Vacancies,1.000000
4,Maximum Vacancies,999.000000


In [51]:
# Convert salary fields to numeric format

job_bank_clean["salary_minimum"] = pd.to_numeric(
    job_bank_clean["salary_minimum"],
    errors="coerce"
)

job_bank_clean["salary_maximum"] = pd.to_numeric(
    job_bank_clean["salary_maximum"],
    errors="coerce"
)

In [53]:
# Calculate average salary from reported minimum and maximum salary ranges

job_bank_clean["salary_average"] = (
    job_bank_clean["salary_minimum"] +
    job_bank_clean["salary_maximum"]
) / 2

In [55]:
# Save preliminary cleaned Job Bank dataset

CLEAN_FILE = (
    PROJECT_ROOT /
    "Data" /
    "Processed" /
    "job_bank_preliminary_clean.csv"
)

CLEAN_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

job_bank_clean.to_csv(
    CLEAN_FILE,
    index=False
)

print(
    "Preliminary cleaned dataset saved to:",
    CLEAN_FILE
)

Preliminary cleaned dataset saved to: C:\Users\Admin\Capstone_Project\Data\Processed\job_bank_preliminary_clean.csv


## Data Quality Summary

The Job Bank dataset was examined to assess its suitability for labour market analysis and subsequent analytical activities.

Key findings include:

- Dataset dimensions were reviewed to understand the overall size and structure.
- Data types were verified and standardized where required.
- Missing values were identified and assessed for each variable.
- Duplicate records were evaluated to understand potential data duplication issues..
- Geographic coverage includes multiple Canadian provinces and territories.
- NOC classifications were reviewed to understand occupational coverage.
- Wage variables, vacancy information, education requirements, experience requirements, and employment characteristics were assessed.
- A cleaned version of the dataset was created and saved as **job_bank_preliminary_clean.csv**, which will be used for Week 2 descriptive analytics and Week 3 diagnostic analysis.